# GeoNUSAF — colourise the Stage-1 generated masks

Stage 1 writes its samples in the project interchange format: single-channel PNGs whose
pixel **values are class indices 0–5**, plus 255 for ignore. An 8-bit viewer stretches
0–255 to black–white, so values of 0, 1, 2 … all land inside the first 2 % of the ramp
and every tile reads as black. The files are correct — they are just not meant to be
looked at directly.

This notebook pulls the archive from the Hub, extracts it, maps the indices onto the
**same palette the real dataset masks use**, previews the result inline, and leaves a zip
in the working directory.

| section | what it does |
|---|---|
| 1 | environment and imports |
| 2 | `CFG` |
| 3 | paths |
| 4 | palette and lookup table |
| 5 | fetch the archive |
| 6 | extract |
| 7 | mask loader + contract check |
| 8 | colourise every mask |
| 9 | per-file class shares |
| 10 | pool summary against the real fold-1 shares |
| 11 | legend and contact sheets |
| 12 | inline preview |
| 13 | zip and download |

Internet must be **ON** for section 5. Nothing here needs a GPU; the whole run is a few
minutes on CPU.

## 1. Environment and imports

In [ ]:
import importlib, subprocess, sys

for mod, pkg in [("huggingface_hub", "huggingface_hub"), ("PIL", "Pillow")]:
    if importlib.util.find_spec(mod) is None:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        if r.returncode:
            print(f"WARNING: could not install {pkg} -- section 5 needs it only for the "
                  f"Hub download; set cfg.LOCAL_ARCHIVE to work from a local file instead")

import hashlib, os, re, shutil, tarfile, time, zipfile
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

print("numpy", np.__version__, "| pandas", pd.__version__)

## 2. CFG

One dataclass, one source of truth. `LOCAL_ARCHIVE` skips the download when the tarball
is already on disk (a Kaggle input, a previous run, a manual upload).

In [ ]:
@dataclass
class CFG:
    # --- source -----------------------------------------------------------
    HF_REPO: str = "sugam24/geonusaf-stage1-maskddpm-onehot-block-fold1"
    HF_FILE: str = "gen_masks_512_block_fold1.tar.gz"
    HF_REPO_TYPE: str = "model"      # the masks sit in the S1 model repo
    LOCAL_ARCHIVE: str = ""          # non-empty -> use this path, skip the download

    # --- output -----------------------------------------------------------
    OUT_DIR: str = "colored_masks"
    ZIP_STEM: str = "colored_masks"

    # --- rendering --------------------------------------------------------
    CHANNEL_ORDER: str = "RGB"       # "RGB" for viewers, "BGR" only for cv2 round-trips
    GRID_N: int = 16                 # masks per contact sheet
    GRIDS_MAX: int = 6               # how many contact sheets to write
    THUMB: int = 256                 # thumbnail edge in the contact sheets
    PREVIEW_N: int = 8               # how many to show inline in section 12
    LIMIT: int = 0                   # 0 = every mask, else first N (quick test)

cfg = CFG()

# escape hatch used when this notebook is executed head-less for testing
if os.environ.get("GEONUSAF_LOCAL_ARCHIVE"):
    cfg.LOCAL_ARCHIVE = os.environ["GEONUSAF_LOCAL_ARCHIVE"]

for k, v in asdict(cfg).items():
    print(f"{k:<16} {v}")

## 3. Paths

In [ ]:
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / cfg.OUT_DIR
PNG_DIR, GRID_DIR = OUT / "png", OUT / "grids"
EXTRACT = WORK / "_extracted_masks"

for d in (PNG_DIR, GRID_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("work    :", WORK)
print("output  :", OUT)

## 4. Palette and lookup table

The notebooks record this palette in **BGR** because they read masks with `cv2.imread`.
A PNG actually stores RGB, so that is what gets written here; read a file back with
`cv2.imread` and the familiar BGR tuples come out unchanged.

| index | class | RGB written | BGR as cv2 prints it |
|---|---|---|---|
| 0 | Residential | `(0, 0, 128)` | `(128, 0, 0)` |
| 1 | Road | `(0, 128, 0)` | `(0, 128, 0)` |
| 2 | River | `(128, 0, 0)` | `(0, 0, 128)` |
| 3 | Forest | `(128, 128, 0)` | `(0, 128, 128)` |
| 4 | UnusedLand | `(0, 128, 128)` | `(128, 128, 0)` |
| 5 | Agricultural | `(128, 0, 128)` | `(128, 0, 128)` |
| 255 | ignore | `(0, 0, 0)` | `(0, 0, 0)` |

Residential↔River and Forest↔UnusedLand are exact channel-order mirrors, so a flipped
order still looks plausible — it silently swaps the rarest class with the commonest. A
real Kathmandu tile is mostly navy; if the output comes out mostly maroon, the order is
reversed.

Any value outside {0…5, 255} is rendered **magenta** and counted, so a contract violation
is loud instead of silent.

In [ ]:
CLASS_NAMES = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]
N_CLASSES, IGNORE_INDEX, UNKNOWN_RGB = 6, 255, (255, 0, 255)

PALETTE_RGB = {
    0: (0, 0, 128),      # Residential  navy
    1: (0, 128, 0),      # Road         green
    2: (128, 0, 0),      # River        maroon
    3: (128, 128, 0),    # Forest       olive
    4: (0, 128, 128),    # UnusedLand   teal
    5: (128, 0, 128),    # Agricultural purple
    IGNORE_INDEX: (0, 0, 0),
}

# real fold-1 TRAIN shares, for the drift check in section 10
REAL_FOLD1_SHARES = {"Residential": 52.53, "Road": 1.431, "River": 0.469,
                     "Forest": 12.40, "UnusedLand": 14.27, "Agricultural": 18.51,
                     "ignore": 0.392}


def build_lut(order="RGB"):
    lut = np.full((256, 3), UNKNOWN_RGB, dtype=np.uint8)
    for idx, rgb in PALETTE_RGB.items():
        lut[idx] = rgb[::-1] if order.upper() == "BGR" else rgb
    return lut


LUT = build_lut(cfg.CHANNEL_ORDER)
print(pd.DataFrame({"index": list(range(N_CLASSES)) + [IGNORE_INDEX],
                    "class": CLASS_NAMES + ["ignore"],
                    "written": [tuple(int(v) for v in LUT[i])
                                for i in list(range(N_CLASSES)) + [IGNORE_INDEX]]}).to_string(index=False))

## 5. Fetch the archive

`hf_hub_download` caches, so re-running this cell costs nothing after the first pull. If
the Hub is unreachable, set `cfg.LOCAL_ARCHIVE` in section 2 to a path — an attached
Kaggle dataset, or a manual upload — and re-run from there.

In [ ]:
if cfg.LOCAL_ARCHIVE:
    ARCHIVE = Path(cfg.LOCAL_ARCHIVE)
    assert ARCHIVE.exists(), f"LOCAL_ARCHIVE not found: {ARCHIVE}"
    print("using local archive")
else:
    from huggingface_hub import hf_hub_download
    t0 = time.time()
    ARCHIVE = Path(hf_hub_download(repo_id=cfg.HF_REPO, filename=cfg.HF_FILE,
                                   repo_type=cfg.HF_REPO_TYPE))
    print(f"downloaded in {time.time() - t0:.1f} s")

print("archive :", ARCHIVE)
print("size    :", f"{ARCHIVE.stat().st_size / 1e6:.1f} MB")
print("sha256  :", hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()[:16])

## 6. Extract

In [ ]:
IMG_EXT = {".png", ".tif", ".tiff", ".bmp"}


def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]


if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)

if ARCHIVE.suffix == ".zip":
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(EXTRACT)
else:
    with tarfile.open(ARCHIVE, "r:*") as tf:
        safe = [m for m in tf.getmembers()
                if not m.name.startswith("/") and ".." not in Path(m.name).parts]
        tf.extractall(EXTRACT, members=safe)

FILES = sorted((p for p in EXTRACT.rglob("*") if p.suffix.lower() in IMG_EXT), key=natural_key)
if not FILES:
    FILES = sorted(EXTRACT.rglob("*.npy"), key=natural_key)
if cfg.LIMIT:
    FILES = FILES[:cfg.LIMIT]

assert FILES, f"no mask files found under {EXTRACT}"
print(f"{len(FILES)} mask files")
print("first    :", FILES[0].relative_to(EXTRACT))
print("last     :", FILES[-1].relative_to(EXTRACT))

## 7. Mask loader and contract check

Handles every shape the archive might hold: plain single-channel PNGs, palettised PNGs
(indices live in the palette slot, not the RGB values), grayscale saved as three equal
channels, `.npy` arrays, and already-coloured masks — which get mapped back to indices by
whichever channel order leaves fewer unexplained pixels.

In [ ]:
def _map_with_order(rgb, reverse):
    out = np.full(rgb.shape[:2], IGNORE_INDEX, dtype=np.uint8)
    matched = np.all(rgb == 0, axis=-1)
    for idx, col in PALETTE_RGB.items():
        if idx == IGNORE_INDEX:
            continue
        cand = col[::-1] if reverse else col
        hit = np.all(rgb == np.array(cand, dtype=rgb.dtype), axis=-1)
        out[hit] = idx
        matched |= hit
    return out, int((~matched).sum())


def rgb_to_index(rgb):
    fwd, miss_f = _map_with_order(rgb, False)
    rev, miss_r = _map_with_order(rgb, True)
    return fwd if miss_f <= miss_r else rev


def load_index_mask(path):
    if path.suffix.lower() == ".npy":
        a = np.load(path)
        return (a[0] if a.ndim == 3 and a.shape[0] == 1 else a).astype(np.uint8)
    im = Image.open(path)
    if im.mode == "P":
        return np.array(im, dtype=np.uint8)
    a = np.array(im)
    if a.ndim == 2:
        return a.astype(np.uint8)
    a = a[..., :3]
    if np.array_equal(a[..., 0], a[..., 1]) and np.array_equal(a[..., 1], a[..., 2]):
        return a[..., 0].astype(np.uint8)
    return rgb_to_index(a)


probe = load_index_mask(FILES[0])
vals = np.unique(probe)
print("shape          :", probe.shape, probe.dtype)
print("values present :", vals.tolist())
print("legal          :", set(vals.tolist()) <= set(range(N_CLASSES)) | {IGNORE_INDEX})

# proof that the colour map round-trips exactly
assert np.array_equal(rgb_to_index(build_lut("RGB")[probe]), probe)
print("round-trip     : exact")

## 8. Colourise every mask

In [ ]:
rows, color_paths = [], []
pool_counts = np.zeros(256, dtype=np.int64)
presence = np.zeros(N_CLASSES, dtype=np.int64)

t0 = time.time()
for i, f in enumerate(FILES, 1):
    idx = load_index_mask(f)
    dst = PNG_DIR / (f.stem + ".png")
    Image.fromarray(LUT[idx]).save(dst)
    color_paths.append(dst)

    counts = np.bincount(idx.ravel(), minlength=256)
    pool_counts += counts
    presence += (counts[:N_CLASSES] > 0).astype(np.int64)
    total = idx.size
    known = counts[:N_CLASSES].sum() + counts[IGNORE_INDEX]
    rows.append({"file": f.name, "h": idx.shape[0], "w": idx.shape[1],
                 **{CLASS_NAMES[c]: 100.0 * counts[c] / total for c in range(N_CLASSES)},
                 "ignore": 100.0 * counts[IGNORE_INDEX] / total,
                 "illegal_pct": 100.0 * (total - known) / total})

    if i % 200 == 0 or i == len(FILES):
        print(f"  {i}/{len(FILES)}  ({time.time() - t0:.1f} s)")

print(f"\nwrote {len(color_paths)} coloured PNGs to {PNG_DIR}")

## 9. Per-file class shares

In [ ]:
shares = pd.DataFrame(rows).round(4)
shares.to_csv(OUT / "class_shares.csv", index=False)
print("saved", OUT / "class_shares.csv")
shares.head(10)

## 10. Pool summary against the real fold-1 shares

The raw Stage-1 pool is expected to drift from the real marginal — before post-processing
and greedy matching, River appeared in 30–34 % of generated masks against 16.8 % of real
ones. If this table shows that, it is the known generator behaviour, not a colourisation
fault.

In [ ]:
tot = pool_counts.sum()
gen = {CLASS_NAMES[c]: 100.0 * pool_counts[c] / tot for c in range(N_CLASSES)}
gen["ignore"] = 100.0 * pool_counts[IGNORE_INDEX] / tot

summary = pd.DataFrame({
    "generated_%": pd.Series(gen),
    "real_fold1_%": pd.Series(REAL_FOLD1_SHARES),
})
summary["ratio"] = (summary["generated_%"] / summary["real_fold1_%"]).round(3)
summary["present_in"] = pd.Series({CLASS_NAMES[c]: f"{presence[c]}/{len(FILES)}"
                                   for c in range(N_CLASSES)}).reindex(summary.index, fill_value="-")
summary = summary.round(4)

illegal = 100.0 * (tot - pool_counts[:N_CLASSES].sum() - pool_counts[IGNORE_INDEX]) / tot
bad_files = int((shares["illegal_pct"] > 0).sum())

lines = [f"files              : {len(FILES)}",
         f"channel order      : {cfg.CHANNEL_ORDER}",
         f"illegal pixels     : {illegal:.6f} %",
         f"files with illegal : {bad_files}", "",
         summary.to_string()]
(OUT / "summary.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## 11. Legend and contact sheets

In [ ]:
def _font(size):
    for cand in ("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"):
        if os.path.exists(cand):
            try:
                return ImageFont.truetype(cand, size)
            except Exception:
                pass
    return ImageFont.load_default()


def make_legend(path, swatch=132):
    entries = [(c, CLASS_NAMES[c]) for c in range(N_CLASSES)] + [(IGNORE_INDEX, "ignore")]
    pad, gap = 28, 16
    w = pad * 2 + len(entries) * swatch + (len(entries) - 1) * gap
    img = Image.new("RGB", (w, pad * 2 + swatch + 80), (255, 255, 255))
    d = ImageDraw.Draw(img)
    d.text((pad, 8), "GeoNUSAF land-use palette (6 classes + ignore)",
           fill=(20, 20, 20), font=_font(22))
    x, y = pad, pad + 22
    for idx, name in entries:
        col = tuple(int(v) for v in LUT[idx])
        d.rectangle([x, y, x + swatch, y + swatch], fill=col, outline=(60, 60, 60))
        d.text((x, y + swatch + 6), f"{idx}  {name}", fill=(20, 20, 20), font=_font(15))
        d.text((x, y + swatch + 24), f"{cfg.CHANNEL_ORDER} {col[0]},{col[1]},{col[2]}",
               fill=(110, 110, 110), font=_font(13))
        x += swatch + gap
    img.save(path)


def legend_strip(width, height=46):
    strip = Image.new("RGB", (width, height), (255, 255, 255))
    d = ImageDraw.Draw(strip)
    cell, box = width // N_CLASSES, 20
    for c in range(N_CLASSES):
        x = c * cell + 8
        d.rectangle([x, height // 2 - box // 2, x + box, height // 2 + box // 2],
                    fill=tuple(int(v) for v in LUT[c]), outline=(60, 60, 60))
        d.text((x + box + 6, height // 2 - 8), CLASS_NAMES[c],
               fill=(20, 20, 20), font=_font(14))
    return strip


def make_grid(paths, out_path, thumb, titles=True):
    cols = int(np.ceil(np.sqrt(len(paths))))
    r_n = int(np.ceil(len(paths) / cols))
    cap, pad, strip_h = 20 if titles else 0, 6, 46
    W = cols * thumb + (cols + 1) * pad
    H = r_n * (thumb + cap) + (r_n + 1) * pad
    sheet = Image.new("RGB", (W, H + strip_h), (255, 255, 255))
    d = ImageDraw.Draw(sheet)
    for i, p in enumerate(paths):
        r, c = divmod(i, cols)
        x, y = pad + c * (thumb + pad), pad + r * (thumb + cap + pad)
        sheet.paste(Image.open(p).convert("RGB").resize((thumb, thumb), Image.NEAREST), (x, y))
        if titles:
            d.text((x + 2, y + thumb + 3), p.stem[:34], fill=(40, 40, 40), font=_font(13))
    sheet.paste(legend_strip(W, strip_h), (0, H))
    sheet.save(out_path)


make_legend(OUT / "legend.png")
n_sheets = min(cfg.GRIDS_MAX, int(np.ceil(len(color_paths) / cfg.GRID_N)))
for g in range(n_sheets):
    chunk = color_paths[g * cfg.GRID_N:(g + 1) * cfg.GRID_N]
    make_grid(chunk, GRID_DIR / f"grid_{g:02d}.png", cfg.THUMB)
print(f"legend.png + {n_sheets} contact sheet(s)")

## 12. Inline preview

In [ ]:
sel = color_paths[:cfg.PREVIEW_N]
cols = min(4, len(sel))
r_n = int(np.ceil(len(sel) / cols))
fig, axes = plt.subplots(r_n, cols, figsize=(4 * cols, 4 * r_n))
for ax, p in zip(np.atleast_1d(axes).ravel(), sel):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem, fontsize=9)
    ax.axis("off")
for ax in np.atleast_1d(axes).ravel()[len(sel):]:
    ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, fc=np.array(LUT[c]) / 255) for c in range(N_CLASSES)]
fig.legend(handles, CLASS_NAMES, loc="lower center", ncol=6, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

## 13. Zip and download

On Kaggle the zip lands in `/kaggle/working` — grab it from the Output panel on the right,
or from the file link this cell prints in Jupyter/Colab.

In [ ]:
ZIP_PATH = WORK / f"{cfg.ZIP_STEM}.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(OUT.parent))

print(f"{ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.1f} MB, "
      f"{len(zipfile.ZipFile(ZIP_PATH).namelist())} files)")

try:  # Colab
    from google.colab import files as _colab_files
    _colab_files.download(str(ZIP_PATH))
except Exception:
    try:  # Jupyter / Kaggle notebook link
        from IPython.display import FileLink, display
        display(FileLink(str(ZIP_PATH.relative_to(Path.cwd()))
                         if ZIP_PATH.is_relative_to(Path.cwd()) else str(ZIP_PATH)))
    except Exception:
        print("download it from the working directory")